# Lab 3 Part 2

This is an individual assignment.

---

Write your own code. You may repurpose any functions built during lecture. You may use ```scikit-learn``` functions.

---

# Exercise 1 (30 points)

**In this question, you will be working with the [marathon time predictions dataset](https://www.kaggle.com/datasets/girardi69/marathon-time-predictions). (Same dataset as in Lab 2).**

**Carry hyperparameter tuning for training 3 regression models:**

* **_Model 1_: (multiple) linear regression with Lasso regularizer.**

* **_Model 2_: polynomial regression  with Lasso regularizer.**

* **_Model 3_: random forest.**

**For model 1, experiment with the regularizer term $\lambda$.**

**For model 2, experiment with polynomial degree, the ```interaction_only``` argument and the regularizer term $\lambda$.**

**For model 3, experiment with the number of trees and the ```criterion```.**


1. (8 points) **Build a ```GridSearchCV``` for each model using ```scikit-learn``` pipelines.**

2. (8 points) **Build a ```RandomizedSearchCV``` for each model using ```scikit-learn``` pipelines.**

    * **Consider an exponential distribution for the regularizer term $\lambda$.**
    * **Consider a uniform distribution for the number of trees.**

3. (8 points) **Which set of hyperparmeters worked best for each model?**

4. (6 points) **Train the final models and save them as ```pickle``` files.**

## Data preprocessing

In [98]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV ,RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

import numpy as np
from scipy.stats import expon, randint
import pickle



import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('bmh')

In [3]:
df = pd.read_csv("MarathonData.csv")
df

,id,Marathon,Name,Category,km4week,sp4week,CrossTraining,Wall21,MarathonTime,CATEGORY
0,1,Prague17,Blair MORGAN,MAM,132.8,14.434783,NaN,1.16,2.37,A
1,2,Prague17,Robert Heczko,MAM,68.6,13.674419,NaN,1.23,2.59,A
2,3,Prague17,Michon Jerome,MAM,82.7,13.520436,NaN,1.30,2.66,A
3,4,Prague17,Daniel Or lek,M45,137.5,12.258544,NaN,1.32,2.68,A
4,5,Prague17,Luk ? Mr zek,MAM,84.6,13.945055,NaN,1.36,2.74,A
...,...,...,...,...,...,...,...,...,...,...
82,83,Prague17,Stefano Vegliani,M55,50.0,10.830325,NaN,2.02,3.93,D
83,84,Prague17,Andrej Madliak,M40,33.6,10.130653,ciclista 3h,1.94,3.93,D
84,85,Prague17,Yoi Ohsako,M40,55.4,11.043189,NaN,1.94,3.94,D
85,86,Prague17,Simon Dunn,M45,33.2,11.066667,NaN,2.05,3.95,D


In [23]:
encoder = OneHotEncoder(drop='first', sparse_output=False)
df_noname = df.drop(['Marathon', 'Name'], axis=1)
df_noname

,id,Category,km4week,sp4week,CrossTraining,Wall21,MarathonTime,CATEGORY
0,1,MAM,132.8,14.434783,NaN,1.16,2.37,A
1,2,MAM,68.6,13.674419,NaN,1.23,2.59,A
2,3,MAM,82.7,13.520436,NaN,1.30,2.66,A
3,4,M45,137.5,12.258544,NaN,1.32,2.68,A
4,5,MAM,84.6,13.945055,NaN,1.36,2.74,A
...,...,...,...,...,...,...,...,...
82,83,M55,50.0,10.830325,NaN,2.02,3.93,D
83,84,M40,33.6,10.130653,ciclista 3h,1.94,3.93,D
84,85,M40,55.4,11.043189,NaN,1.94,3.94,D
85,86,M45,33.2,11.066667,NaN,2.05,3.95,D


In [26]:
numeric_columns = df_noname.select_dtypes(exclude=['object']).columns
categorical_columns = df_noname.select_dtypes(include=['object']).columns

encoded_data = encoder.fit_transform(df_noname[categorical_columns])
encoded_feature_names = encoder.get_feature_names_out(categorical_columns)

encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoded_feature_names,
    index=df_noname.index
)

df_clean = pd.concat([
    df_noname[numeric_columns],
    encoded_df
], axis=1)

In [27]:
df_clean

,id,km4week,sp4week,MarathonTime,Category_M45,Category_M50,Category_M55,Category_MAM,Category_WAM,Category_nan,...,Wall21_1.90,Wall21_1.93,Wall21_1.94,Wall21_1.97,Wall21_1.98,Wall21_2.02,Wall21_2.05,CATEGORY_B,CATEGORY_C,CATEGORY_D
0,1,132.8,14.434783,2.37,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,68.6,13.674419,2.59,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,82.7,13.520436,2.66,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,137.5,12.258544,2.68,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,84.6,13.945055,2.74,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,83,50.0,10.830325,3.93,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
83,84,33.6,10.130653,3.93,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
84,85,55.4,11.043189,3.94,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
85,86,33.2,11.066667,3.95,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


In [28]:
test , train = train_test_split( df_clean , test_size=0.25,  random_state=2025 ,shuffle=True)

In [29]:
x_train = train.drop(labels='MarathonTime', axis=1)
x_test = test.drop(labels='MarathonTime', axis=1) 
y_train = train['MarathonTime'].copy()
y_test = test['MarathonTime'].copy()


In [30]:
x_test.shape, x_train.shape, y_test.shape, y_train.shape

((65, 68), (22, 68), (65,), (22,))

## MODEL_1

In [55]:
alpha = [0.01,0.1,1,10,100,1000,10000]

In [56]:
for i in alpha: 
    model_1 = Lasso(
    alpha = i ,  
    random_state=2025 )
    model_1.fit(x_train, y_train)
    y_pred = model_1.predict(x_test)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"lambda = {i:7}, mse:{mse:10.4f},r2:{r2:4f}")

lambda =    0.01, mse:  331.2083,r2:-2586.676451
lambda =     0.1, mse:    0.0038,r2:0.969932
lambda =       1, mse:    0.0021,r2:0.983680
lambda =      10, mse:    0.1098,r2:0.142019
lambda =     100, mse:    0.1288,r2:-0.006125
lambda =    1000, mse:    0.1288,r2:-0.006125
lambda =   10000, mse:    0.1288,r2:-0.006125


## MODEL_2

In [60]:
degrees = [2, 3, 4]
interactions = [True, False]
alphas = [0.0001, 0.001, 0.01, 0.1, 1, 10]
results = []

In [64]:
for degree in degrees:
    for interaction_only in interactions:
        for alpha in alphas:
            poly = PolynomialFeatures(
                degree=degree,
                interaction_only=interaction_only
            )
            x_train_poly = poly.fit_transform(x_train)
            x_test_poly = poly.transform(x_test)

            model = Lasso(alpha=alpha, random_state=2025)
            model.fit(x_train_poly, y_train)
                     
            y_pred = model.predict(x_test_poly)
            r2 = r2_score(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
             
            results.append({
                'Degree': degree,
                'Interaction_Only': interaction_only,
                'Alpha': alpha,
                'R2': r2, 
                'MSE': mse,
            })

c:\Users\champ\miniconda3\envs\aml\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.778e-04, tolerance: 3.886e-04
  model = cd_fast.enet_coordinate_descent(
c:\Users\champ\miniconda3\envs\aml\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.536e-04, tolerance: 3.886e-04
  model = cd_fast.enet_coordinate_descent(
c:\Users\champ\miniconda3\envs\aml\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

---

In [70]:
results_df = pd.DataFrame(results)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
pd.set_option('display.max_rows', None)  
results_df


,Degree,Interaction_Only,Alpha,R2,MSE
0,2,True,0.0001,-7593.4128,972.0428
1,2,True,0.0010,-5953.7191,762.1711
2,2,True,0.0100,-32761.1867,4193.3784
3,2,True,0.1000,-42495.6485,5439.3356
4,2,True,1.0000,-78700.0244,10073.2952
5,2,True,10.0000,-27523.1969,3522.9448
6,2,False,0.0001,-3132091057.5242,400890308.5252
7,2,False,0.0010,-1340521193.1023,171579288.4200
8,2,False,0.0100,-6341.0991,811.7536
9,2,False,0.1000,-3587.6024,459.3212


In [71]:
pd.reset_option('all')

C:\Users\champ\AppData\Local\Temp\ipykernel_20956\2786130087.py:1: FutureWarning: data_manager option is deprecated and will be removed in a future version. Only the BlockManager will be available.
  pd.reset_option('all')
C:\Users\champ\AppData\Local\Temp\ipykernel_20956\2786130087.py:1: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  pd.reset_option('all')


## Model_3

In [79]:
pipeline = Pipeline([
    ('rf', RandomForestRegressor(random_state=2025))
])
rf_param_grid = {
    'rf__n_estimators': [100, 200, 300, 400, 500],
    'rf__criterion': ['squared_error', 'absolute_error']
}
rf_grid = GridSearchCV(
    pipeline,
    rf_param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)


rf_grid.fit(x_train, y_train)

print("Best parameters (Grid Search):", rf_grid.best_params_)
print("Best score (Grid Search):", -rf_grid.best_score_)

Best parameters (Grid Search): {'rf__criterion': 'squared_error', 'rf__n_estimators': 100}
Best score (Grid Search): 0.01461030060000012


In [81]:
rf_param_dist = {
    'rf__n_estimators': randint(100, 1000), 
    'rf__criterion': ['squared_error', 'absolute_error']
}

rf_random = RandomizedSearchCV(
    pipeline,
    rf_param_dist,
    n_iter=20,  
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=2025
)

rf_random.fit(x_train, y_train)

print("Best parameters (Random Search):", rf_random.best_params_)
print("Best score (Random Search):", -rf_random.best_score_)

Best parameters (Random Search): {'rf__criterion': 'squared_error', 'rf__n_estimators': 962}
Best score (Random Search): 0.015519350257822561


In [82]:
best_params = rf_random.best_params_  
final_rf = RandomForestRegressor(
    n_estimators=best_params['rf__n_estimators'],
    criterion=best_params['rf__criterion'],
    random_state=42
)
final_rf.fit(x_train, y_train)

with open('random_forest_model.pkl', 'wb') as file:
    pickle.dump(final_rf, file)

# Exercise 2 (20 points)

### 1. (5 points) **Load your (best) trained models (total of 3) and evaluate the performance in the test set.**


In [95]:
model_1 = Lasso(
    alpha = 1 ,  
    random_state=2025)

model_1.fit(x_train, y_train)
y_pred_1 = model_1.predict(x_test)
mse = mean_squared_error(y_test, y_pred_1)
r2 = r2_score(y_test, y_pred_1)
print("Model_1:",mse,r2)

Model_1: 0.002088859813218768 0.9836800765586948


In [96]:
poly = PolynomialFeatures(
                degree=2,
                interaction_only=False
            )
x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

model = Lasso(alpha=10, random_state=2025)
model.fit(x_train_poly, y_train)
                     
y_pred_2 = model.predict(x_test_poly)
r2 = r2_score(y_test, y_pred_2)
mse = mean_squared_error(y_test, y_pred_2)
print("Model_2:",mse,r2)        

Model_2: 0.00492675266134735 0.9615080793174251


In [97]:
best_rf = rf_grid.best_estimator_
y_pred_3 = best_rf.predict(x_test)

test_r2 = r2_score(y_test, y_pred_3)
mse = mean_squared_error(y_test, y_pred_3)
print("Model_3:",mse,r2) 

Model_3: 0.006099581384615391 0.9615080793174251


### 2. (5 points) **Report the RMSE performance and the 95% CI for each.**

In [104]:
def calculate_ci(y_true, y_pred, n_bootstrap=1000, ci=0.95):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    residuals = y_true - y_pred
    bootstrap_rmse = np.zeros(n_bootstrap)
    n_samples = len(y_true)
    
    for i in range(n_bootstrap):

        bootstrap_indices = np.random.randint(0, n_samples, size=n_samples)
        sampled_residuals = residuals[bootstrap_indices]
        
        bootstrap_predictions = y_pred + sampled_residuals
        rmse = np.sqrt(mean_squared_error(y_true, bootstrap_predictions))
        bootstrap_rmse[i] = rmse

    lower = np.percentile(bootstrap_rmse, (1-ci)/2 * 100)
    upper = np.percentile(bootstrap_rmse, (1+ci)/2 * 100)
    
    return lower, upper


In [105]:
for name, predictions in [("Model 1", y_pred_1), 
                        ("Model 2", y_pred_2), 
                        ("Model 3", y_pred_3)]:
    
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    ci_lower, ci_upper = calculate_ci(y_test, predictions)
    
    print(f"\n{name}:")
    print(f"RMSE: {rmse:.4f}")
    print(f"95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")


Model 1:
RMSE: 0.0457
95% CI: [0.0384, 0.0510]

Model 2:
RMSE: 0.0702
95% CI: [0.0789, 0.1071]

Model 3:
RMSE: 0.0781
95% CI: [0.0872, 0.1137]


### 3. (5 points) **Based on these results, which model would you select?**

I'll pick Model_1, because it achieves the best predictive performance with the lowest RMSE